# IndicF5 — Indian English TTS (Zero-Shot + Fine-Tune)

**MIT license** — code AND weights. Built specifically for Indian voices by AI4Bharat.

| Step | What | Time | Action After |
|------|------|------|-------------|
| **1a** | Install all packages | ~5 min | **RESTART RUNTIME** |
| **1b** | Setup + login + data download | ~10 min | — |
| **2** | Load IndicF5 model | ~5 min | Model loads? |
| **3** | Pick reference voices | ~2 min | — |
| **4** | Generate podcast (NO training) | ~5 min | **Good enough?** |
| **5** | Fine-tune (optional) | ~1-2 hrs | Better? |
| **6** | Save to Drive | ~5 min | Done! |

**IMPORTANT:** Step 1a installs packages then you MUST restart the runtime before Step 1b.

## Step 1a: Install Packages (then RESTART runtime)

Run this cell, wait for it to finish, then click **Runtime > Restart runtime**.

In [ ]:
# Install IndicF5 and dependencies
# This will downgrade numpy — that's OK, we fix it at the end
!apt-get -qq install -y ffmpeg > /dev/null 2>&1
!pip install -q 'transformers>=4.45,<4.50'
!pip install -q git+https://github.com/ai4bharat/IndicF5.git

# Force numpy back to 2.x (Colab needs it, IndicF5 wrongly downgrades it)
!pip install -q 'numpy>=2.0' --force-reinstall 2>&1 | tail -1

import numpy as np
print(f"\nnumpy version: {np.__version__}")
print("\n" + "=" * 50)
print("  INSTALL DONE.")
print("  NOW CLICK: Runtime > Restart runtime")
print("  Then run Step 1b (skip this cell on re-run)")
print("=" * 50)

import os, torch, shutil, glob
import numpy as np
print(f"numpy: {np.__version__} (must be 2.x)")
assert int(np.__version__.split('.')[0]) >= 2, "numpy is not 2.x! Re-run Step 1a and restart."

# --- GPU Check ---
assert torch.cuda.is_available(), "Need GPU! Runtime > Change runtime type > A100"
print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- HuggingFace Login ---
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from huggingface_hub import HfApi
try:
    print(f"HF user: {HfApi().whoami()['name']}")
except Exception:
    print("Not logged into HuggingFace!")
    from huggingface_hub import login
    login()

# --- Mount Google Drive (for auto-backup) ---
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BACKUP = '/content/drive/MyDrive/indian_tts_indicf5'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
print(f"Drive backup: {DRIVE_BACKUP}")

# --- Download Svarah Indian English Data ---
os.chdir('/content')
if not os.path.exists('/content/indian_tts'):
    !git clone https://github.com/seetha0712/text2speech_1.git /content/indian_tts
os.chdir('/content/indian_tts')
!git checkout claude/custom-indian-tts-model-TUAjJ -q
!git pull origin claude/custom-indian-tts-model-TUAjJ -q
!pip install -q num2words 2>&1 | tail -1
!pip install -q -e . 2>&1 | tail -1
!apt-get install -qq espeak-ng > /dev/null 2>&1

if not os.path.exists('/content/data/train.txt'):
    print("\nDownloading Svarah Indian English data...")
    !python -m indian_tts.data.preprocess --source svarah --output /content/data
else:
    with open('/content/data/train.txt') as f:
        n = sum(1 for l in f if l.strip() and not l.startswith('#'))
    print(f"\nSvarah data already downloaded: {n} training samples")

print("\nStep 1b complete!")

In [ ]:
# Load IndicF5 model
from transformers import AutoModel
import numpy as np
import soundfile as sf

print("Loading IndicF5 model (downloads ~3GB)...")
model = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True)
print("Model loaded!")
print("CHECKPOINT: Model loads successfully.")

## Step 2: Select Reference Voices

In [ ]:
import glob
import IPython.display as ipd

# Find reference clips from Svarah data
male_wavs = sorted(glob.glob('/content/data/svarah/male/svarah_*.wav'))
female_wavs = sorted(glob.glob('/content/data/svarah/female/svarah_*.wav'))

print(f"Male clips: {len(male_wavs)}")
print(f"Female clips: {len(female_wavs)}")

# Pick clips that are 5-12 seconds
def find_good_reference(wav_list, min_dur=5, max_dur=12):
    for path in wav_list:
        data, sr = sf.read(path)
        dur = len(data) / sr
        if min_dur <= dur <= max_dur:
            return path, dur
    return wav_list[0], len(sf.read(wav_list[0])[0]) / sf.read(wav_list[0])[1]

male_ref, male_dur = find_good_reference(male_wavs)
female_ref, female_dur = find_good_reference(female_wavs)

# Get transcripts from manifest
male_ref_text = ""
female_ref_text = ""
for manifest in ['/content/data/train.txt', '/content/data/val.txt', '/content/data/test.txt']:
    if os.path.exists(manifest):
        with open(manifest) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = line.split('|')
                if parts[0] == male_ref:
                    male_ref_text = parts[2]
                if parts[0] == female_ref:
                    female_ref_text = parts[2]

print(f"\nMale ref: {male_ref} ({male_dur:.1f}s)")
print(f"  Text: {male_ref_text[:80]}...")
print(f"Female ref: {female_ref} ({female_dur:.1f}s)")
print(f"  Text: {female_ref_text[:80]}...")

print("\n[MALE reference]")
ipd.display(ipd.Audio(male_ref))
print("[FEMALE reference]")
ipd.display(ipd.Audio(female_ref))

## Step 3: Generate Podcast — Zero-Shot (NO training!)

In [ ]:
import time

PODCAST_SCRIPT = [
    ("female", "Welcome to AI India, the podcast where we explore how artificial intelligence is transforming our country. I am Priya."),
    ("male", "And I am Arjun. Today we are talking about something really exciting. The rise of Indian AI startups."),
    ("female", "That is right, Arjun. India now has over three hundred AI startups, and that number is growing every single month."),
    ("male", "What I find really interesting is that many of these companies are solving uniquely Indian problems. Like agriculture, healthcare in rural areas, and education."),
    ("female", "Absolutely. Take for example an AI system that can detect crop diseases just by looking at a photo taken on a farmer's mobile phone. This is saving thousands of farmers from losing their harvest."),
    ("male", "And in healthcare, AI models are now screening for conditions like diabetic retinopathy and tuberculosis in areas where there are very few doctors available."),
    ("female", "The language barrier is another big challenge that AI is helping with. India has twenty two official languages and hundreds of dialects."),
    ("male", "Exactly. And that is precisely why building speech technology like text to speech systems in Indian languages is so important. We cannot rely only on English."),
    ("female", "Speaking of which, the progress in Indian language AI has been remarkable. Models can now understand and generate speech in Hindi, Tamil, Bengali, and many more."),
    ("male", "The government has also been supportive with initiatives to build open source datasets for Indian languages. This is a game changer."),
    ("female", "So what do you think is next for AI in India, Arjun?"),
    ("male", "I believe we will see AI becoming a part of everyday life. From voice assistants that truly understand Indian accents, to AI tutors that teach children in their mother tongue."),
    ("female", "That is a beautiful vision. And it all starts with building the right foundation, the right data, the right models, and the right talent."),
    ("male", "Could not agree more. India has the talent, and now we are building the tools."),
    ("female", "That is all for today's episode of AI India. Thank you for listening, and we will see you next week."),
    ("male", "Goodbye everyone, and keep innovating!"),
]

os.makedirs('/content/outputs/indicf5_podcast', exist_ok=True)
output_sr = 24000  # IndicF5 outputs at 24kHz

all_segments = []
silence_between = np.zeros(int(output_sr * 0.6))

print("Generating podcast with IndicF5 (zero-shot)...\n")
start = time.time()

for i, (speaker, text) in enumerate(PODCAST_SCRIPT):
    name = "Priya" if speaker == "female" else "Arjun"
    ref_wav = female_ref if speaker == "female" else male_ref
    ref_txt = female_ref_text if speaker == "female" else male_ref_text

    gen_start = time.time()
    audio = model(
        text,
        ref_audio_path=ref_wav,
        ref_text=ref_txt,
    )

    # Normalize audio
    audio = np.array(audio, dtype=np.float32)
    if audio.dtype == np.int16 or np.abs(audio).max() > 1.0:
        audio = audio.astype(np.float32) / max(np.abs(audio).max(), 1.0)

    gen_time = time.time() - gen_start
    duration = len(audio) / output_sr

    print(f"  [{name:5s}] {duration:.1f}s (gen: {gen_time:.1f}s) | {text[:50]}...")

    sf.write(f'/content/outputs/indicf5_podcast/line_{i:02d}_{speaker}.wav', audio, output_sr)

    if i > 0:
        all_segments.append(silence_between)
    all_segments.append(audio)

full_audio = np.concatenate(all_segments)
podcast_path = '/content/outputs/indicf5_podcast/podcast_full.wav'
sf.write(podcast_path, full_audio, output_sr)

total_time = time.time() - start
total_dur = len(full_audio) / output_sr
print(f"\nPodcast generated!")
print(f"  Duration: {total_dur:.0f}s ({total_dur/60:.1f} min)")
print(f"  Generation time: {total_time:.0f}s")
print(f"  RTF: {total_time/total_dur:.2f}x")

In [ ]:
# LISTEN TO THE PODCAST
print("=" * 60)
print("  IndicF5 — Zero-Shot Podcast (NO training)")
print("  Arjun (male) & Priya (female) discuss AI in India")
print("=" * 60)

print("\nFull podcast:")
ipd.display(ipd.Audio('/content/outputs/indicf5_podcast/podcast_full.wav'))

print("\nIndividual lines:")
for i in range(min(4, len(PODCAST_SCRIPT))):
    speaker, text = PODCAST_SCRIPT[i]
    name = "Priya" if speaker == "female" else "Arjun"
    wav_path = f'/content/outputs/indicf5_podcast/line_{i:02d}_{speaker}.wav'
    print(f"\n  [{name}] {text[:60]}...")
    ipd.display(ipd.Audio(wav_path))

# AUTO-BACKUP to Google Drive
shutil.copy2('/content/outputs/indicf5_podcast/podcast_full.wav',
             os.path.join(DRIVE_BACKUP, 'podcast_zero_shot.wav'))
for f in glob.glob('/content/outputs/indicf5_podcast/line_*.wav'):
    shutil.copy2(f, DRIVE_BACKUP)
print(f"\nAuto-backed up to: {DRIVE_BACKUP}")

### DECISION POINT

**Listen above.** Is the quality good enough?

- **YES** -> Skip to Save. You're done!
- **Needs more consistency** -> Proceed to Step 4 (Fine-tuning)

---
## Step 4: Fine-Tune (Optional)

In [ ]:
# Prepare data as CSV for F5-TTS fine-tuning format
import csv

os.makedirs('/content/indicf5_ft', exist_ok=True)

with open('/content/indicf5_ft/metadata.csv', 'w') as f:
    f.write('audio_file|text\n')
    with open('/content/data/train.txt') as train_f:
        for line in train_f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('|')
            audio_path, text = parts[0], parts[2]
            f.write(f'{audio_path}|{text}\n')

# Count lines
with open('/content/indicf5_ft/metadata.csv') as f:
    n_lines = sum(1 for _ in f) - 1
print(f"Prepared {n_lines} utterances for fine-tuning")

In [ ]:
# Process dataset into F5-TTS format
!python -m f5_tts.train.datasets.prepare_csv_wavs \
    /content/indicf5_ft/metadata.csv \
    /content/indicf5_ft/processed

In [ ]:
# Configure accelerate for single GPU
!accelerate config default --mixed_precision fp16

# Fine-tune (~1-2 hrs on A100)
!f5-tts_finetune-cli \
    --exp_name F5TTS_v1_Base \
    --dataset_name indicf5_indian_english \
    --learning_rate 1e-5 \
    --batch_size_per_gpu 6400 \
    --batch_size_type frame \
    --max_samples 64 \
    --grad_accumulation_steps 1 \
    --epochs 50 \
    --num_warmup_updates 200 \
    --save_per_updates 500 \
    --last_per_updates 200 \
    --finetune \
    --logger tensorboard

In [ ]:
# Generate podcast with fine-tuned model
from f5_tts.api import F5TTS

# Find latest checkpoint
ft_ckpts = sorted(glob.glob('/content/ckpts/indicf5_indian_english/model_*.safetensors'))
if ft_ckpts:
    print(f"Using fine-tuned checkpoint: {ft_ckpts[-1]}")
    f5tts = F5TTS(model="F5TTS_v1_Base", ckpt_file=ft_ckpts[-1])

    os.makedirs('/content/outputs/indicf5_ft_podcast', exist_ok=True)
    all_segments = []

    for i, (speaker, text) in enumerate(PODCAST_SCRIPT):
        ref_wav = female_ref if speaker == "female" else male_ref
        ref_txt = female_ref_text if speaker == "female" else male_ref_text
        name = "Priya" if speaker == "female" else "Arjun"

        wav, sr, _ = f5tts.infer(
            ref_file=ref_wav,
            ref_text=ref_txt,
            gen_text=text,
            file_wave=f'/content/outputs/indicf5_ft_podcast/line_{i:02d}_{speaker}.wav',
        )
        audio = np.array(wav, dtype=np.float32).flatten()
        print(f"  [{name:5s}] {len(audio)/sr:.1f}s | {text[:50]}...")
        if i > 0:
            all_segments.append(np.zeros(int(sr * 0.6)))
        all_segments.append(audio)

    full = np.concatenate(all_segments)
    sf.write('/content/outputs/indicf5_ft_podcast/podcast_full.wav', full, sr)
    print("\nFine-tuned podcast:")
    ipd.display(ipd.Audio('/content/outputs/indicf5_ft_podcast/podcast_full.wav'))
else:
    print("No fine-tuned checkpoints found. Run fine-tuning first.")

---
## Save to Google Drive

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

backup_dir = '/content/drive/MyDrive/indian_tts_indicf5'
os.makedirs(backup_dir, exist_ok=True)

for podcast_dir in ['indicf5_podcast', 'indicf5_ft_podcast']:
    src = f'/content/outputs/{podcast_dir}/podcast_full.wav'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(backup_dir, f'{podcast_dir}.wav'))
        print(f"Saved: {podcast_dir}.wav")

print(f"Backup dir: {backup_dir}")

---
## Compare All Three Models

Run this after you've also generated the CosyVoice 2 podcast.

In [ ]:
print("=" * 60)
print("  SIDE-BY-SIDE COMPARISON")
print("=" * 60)

podcasts = {
    'VITS2 (from scratch, 50K steps)': '/content/outputs/podcast_stage_4/podcast_full.wav',
    'CosyVoice 2 (zero-shot)': '/content/outputs/cosyvoice2_podcast/podcast_full.wav',
    'IndicF5 (zero-shot)': '/content/outputs/indicf5_podcast/podcast_full.wav',
    'IndicF5 (fine-tuned)': '/content/outputs/indicf5_ft_podcast/podcast_full.wav',
}

for name, path in podcasts.items():
    if os.path.exists(path):
        data, sr = sf.read(path)
        dur = len(data) / sr
        print(f"\n{name} ({dur:.0f}s):")
        ipd.display(ipd.Audio(path))
    else:
        print(f"\n{name}: Not generated yet")